# SFT evaluation: a checkpoint on the agreed sets

Plays one SFT checkpoint on the evaluation agreed for them (2026-10-30), as two jobs,
one B200 each:

| job | games | passes |
|---|---|---|
| `5games` | the held-out five: pathfinder, taps, cd82, dc22, escape_velocity | 20 |
| `36games` | the official 25 and the 11 ARC-like hard | 3 |

Its scores read against the baseline runs in `arc_3_results/shared/sft_baseline/`, which play
the same sets the same way:

- **Checkpoint:** `arc3-reap307`, the checkpoint the SFT trainer adapts (arc-training #118). A
  checkpoint of the same cut fills the engine's pool alike.
- **Drafter:** main's.
- **Slots:** 152.
- **Baselines:** the multipliers below.

`FP8_KV = True` plays the FP8 KV cache ARC2 plays in place of the seal's INT4 one, at 85 slots
(below). The baselines were played on INT4.

**Set `CHECKPOINT` first.** It has no default, and the first code cell stops with an error
until it names a checkpoint directory on shared storage (`/data/shared/research`); a personal
directory is refused. Runs go under the shared `arc_3_results`; the last cell refuses to submit a
job whose directory is anywhere else.

Copied from `demo.ipynb`. Each run of the notebook launches two new jobs into a new timestamped
directory.

In [ ]:
# --- the checkpoint: required ---------------------------------------------
CHECKPOINT = None  # on shared storage, e.g. "/data/shared/research/checkpoints/<your SFT checkpoint>"
FP8_KV = False  # True: the FP8 KV cache in place of the INT4 one, at the slots it holds

# The settings a student trained on the Kimi teacher traces plays with:
# kimi_student=True turns on the model-visible ones the traces were played
# with. False plays the seal as main builds it, like the baseline_* runs.
KIMI_STUDENT = True

# The agreed evaluation, the 36 at three passes rather than the one first
# agreed (2026-10-30): 108 runs, still under the slots, so every
# game starts at once and the job ends as soon.
HOLDOUT_PASSES = 20
OFFICIAL_PASSES = 3
# As the baseline runs: main's 167 for joint205, scaled to a 307-expert
# pool (seal evidence.md, "joint205's columns"). Under FP8 a token takes
# 1,088 bytes a layer against INT4's 608 (the same note; the B200's pool
# holds 1.72x fewer tokens, ullm flash_optimization.md): 152 x 608/1,088, the
# seal's FP8-era 100 at its window share.
SLOTS = 85 if FP8_KV else 152
# INT4: both jobs end well inside 12 h, as the seal caps a game at 9 h and
# every game of a job starts within its first hour; 12 h also fits the dev
# partition, which refuses a B200 job over 12 h. FP8: both jobs hold more
# games than its slots, so later games start hours in, and only arc's 24 h
# leaves them their 9 h.
WALLTIME_H = 24 if FP8_KV else 12
PARTITION, ACCOUNT = "arc", "arc"  # or "dev", "research" (INT4 only)

import datetime
import getpass
import pathlib

# On shared storage only (2026-10-30), where everyone can read what
# a run played: this cell refuses a checkpoint outside /data/shared/research
# (a personal directory, say), the deploy cell a job dir outside the shared
# results tree.
SHARED_ROOT = pathlib.Path("/data/shared/research")
RESULTS_ROOT = SHARED_ROOT / "arc_3_results"
if CHECKPOINT is None:
    raise ValueError("Set CHECKPOINT to the SFT checkpoint's directory: this notebook has no default.")
# Absolute from here on: the checks below and the engine read the same
# directory, where the workers run from their job's own source dir and would
# find a relative path somewhere else.
CHECKPOINT = str(pathlib.Path(CHECKPOINT).expanduser().resolve())
if not pathlib.Path(CHECKPOINT).is_relative_to(SHARED_ROOT.resolve()):
    raise ValueError(f"CHECKPOINT {CHECKPOINT!r} is not on shared storage ({SHARED_ROOT}); copy it there first.")
if not (pathlib.Path(CHECKPOINT) / "config.json").is_file():
    raise FileNotFoundError(f"CHECKPOINT {CHECKPOINT!r} is not a checkpoint directory (no config.json).")

import taaf.benchmark
import taaf.deploy_slurm
import taaf.game_api
import taaf.standard_benchmarks

import platypus.deployment
from platypus import LLMAgentSolver
from seal import SealAgent

STAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
NAME = pathlib.Path(CHECKPOINT).name
CACHE = "fp8kv" if FP8_KV else "int4kv"
# Beside the baselines, and where the viewer serves them.
RUN_DIR = RESULTS_ROOT / getpass.getuser() / "sft_eval" / f"{NAME}_{CACHE}_{STAMP}"
print(f"checkpoint={CHECKPOINT} kimi_student={KIMI_STUDENT} kv cache={CACHE} slots={SLOTS}")
print(f"run_dir={RUN_DIR}")

## The games

- **Official games** (cd82, dc22 and the rest of the 25) come from TAAF's set at their own
  baselines.
- **escape_velocity and the other ARC-like hard games** are re-arc-3 games, whose baselines are
  optimal solves. They play at three times their baseline, as the baseline runs and the Kimi
  teacher traces do.
- **pathfinder and taps** take their baselines as they are.

TAAF applies a game's `baseline_multiplier` to its baselines when the game starts, so every
score of the run is computed against the scaled numbers, and `benchmark.json` records the factor
for each game run.

In [ ]:
ARC_LIKE_HARD = (
    "anchor_drift-0001", "ball_bounce-0001", "en18-0001", "escape_velocity-0001",
    "glass_orchard-0001", "jc26-0001", "js26-0001", "lantern_moths-0001",
    "marble_loom-0001", "paint_snakes-0001", "talos_beams-0001",
)  # fmt: skip
official = taaf.standard_benchmarks.make_benchmark_official_25().games
arc_like_hard = [taaf.game_api.GameAPI(env_name=g, baseline_multiplier=3.0) for g in ARC_LIKE_HARD]
holdout = [taaf.game_api.GameAPI(env_name=g) for g in ("pathfinder-0001", "taps-0001")]
holdout += [g for g in official if g.env_name in ("cd82-fb555c5d", "dc22-fdcac232")]
holdout += [g for g in arc_like_hard if g.env_name == "escape_velocity-0001"]
assert len(holdout) == 5 and len(official) == 25 and len(arc_like_hard) == 11

SETS = {"5games": (holdout, HOLDOUT_PASSES), "36games": (official + arc_like_hard, OFFICIAL_PASSES)}

# FP8_KV: the KV caches ARC2's duck plays Flash-Next with (main's cell): the
# main cache in FP8 per token and head, the block the engine's alignment
# picks (3,120 is the INT4 layout's), the compressor rings in the shared
# pool; the indexer cache stays FP8. Drafted by the same combined head trained
# at FP8 drafter KV, the drafter's cache following the main one.
ARC23_FP8_DRAFTER = (
    "/data/shared/research/checkpoints/Qwen3.8-Flash-Next-NVFP4-arc23-final-20261029-fp8-drafter-nvfp4-dv16k"
)


def benchmark_for(name: str) -> taaf.benchmark.Benchmark:
    """One job's benchmark: the checkpoint in place of main's, main's drafter."""
    games, passes = SETS[name]
    # The checkpoint into the build, so its engine arguments are built around
    # it: an offline launcher (HF_HUB_OFFLINE) then needs no Hub cache.
    agent = SealAgent.build(environment="slurm-b200", kimi_student=KIMI_STUDENT, checkpoint=CHECKPOINT)
    engine_args = agent.llm_config.backend_factory.engine_args  # main's drafter: speculative_config["model"]
    if FP8_KV:
        engine_args.kv_cache_dtype = "fp8_per_token_head"
        engine_args.block_size = None
        engine_args.transient_kv_pool = False
        engine_args.speculative_config = {**engine_args.speculative_config, "model": ARC23_FP8_DRAFTER}
    agent.run_config.max_concurrent = SLOTS  # the engine's max_num_seqs follows at start
    benchmark = taaf.benchmark.Benchmark(
        label=f"sft_eval_{NAME}_{CACHE}_{name}", games=games, solver=LLMAgentSolver(agent=agent), n_passes=passes
    )
    benchmark.job_dir = RUN_DIR / name
    print(f"{benchmark.label}: {[(g.env_name, g.baseline_multiplier) for g in games]} x {passes} pass(es)")
    print(f"  model={engine_args.model}, kv cache {engine_args.kv_cache_dtype}")
    print(f"  drafter={engine_args.speculative_config.get('model')}")
    return benchmark


benchmarks = {name: benchmark_for(name) for name in SETS}

## The deploy targets

Each job runs on one B200, with `demo.ipynb`'s resources: 24 CPUs and 260 GiB. The 260 GiB
covers the engine, the pinned PLE table, the 64 GiB CPU KV tier the engine claims at start, and
the game processes. The two jobs are submitted one after the other.

In [ ]:
outside = [str(b.job_dir) for b in benchmarks.values() if not b.job_dir.resolve().is_relative_to(RESULTS_ROOT.resolve())]
if outside:
    raise ValueError(f"Job dirs must be under the shared {RESULTS_ROOT}; refusing to submit: {outside}")

for name, benchmark in benchmarks.items():
    target = taaf.deploy_slurm.TufaSlurmTarget(
        partition=PARTITION,
        account=ACCOUNT,
        time=f"{WALLTIME_H}:00:00",
        gpu_count=1,
        cpus_per_gpu=24,
        mem_per_gpu_bytes=260 * 1024**3,
        main_project=platypus.deployment.detect_main_project(),
    )
    handle = await benchmark.deploy(target)
    print(f"{name}: submitted {getattr(handle, 'job_id', None)} -> {handle.job_dir}")

## The baselines, and comparing against them

The reference runs are in `/data/shared/research/arc_3_results/shared/sft_baseline/`, made by
`sft_baseline.ipynb` on `arc3-reap307` with main's drafter, 152 slots and the INT4 KV cache:

| run dir | settings | games | passes |
|---|---|---|---|
| `student_5games` | `kimi_student=True` | the held-out five | 60 |
| `student_36games` | `kimi_student=True` | the official 25 and the 11 ARC-like hard | 4 |
| `baseline_5games` | the seal as main builds it | the held-out five | 60 |
| `baseline_36games` | the seal as main builds it | the official 25 and the 11 ARC-like hard | 4 |

**Which to compare against:** a checkpoint played here (`KIMI_STUDENT = True`) against the
`student_*` runs on the same games. Both notebooks build the same agent and the same games,
multipliers included, so the only difference is the checkpoint. The `baseline_*` runs show what
the student settings change on the untrained model. A run with `FP8_KV = True` differs from them
in the cache too.

**How to compare:** with TAAF's `compare-runs` skill
(`tufa-arc-agi-framework/.claude/skills/compare-runs/SKILL.md`). In Claude Code, ask it to compare
your job dir with the baseline's. The comparison needs both runs finished: a `benchmark.json`
with an `end_time`. Your runs and the baselines sit under different parents, so give the dirs as
an explicit list rather than "the run subfolders of X". The skill's function, with the output
under the shared results tree so its relative links resolve:

```python
import taaf.diagnostics

taaf.diagnostics.regenerate_comparison_diagnostics(
    ["/data/shared/research/arc_3_results/shared/sft_baseline/student_5games", RUN_DIR / "5games"],
    RUN_DIR / "compare_5games",
)
```

The same for `student_36games` and `RUN_DIR / "36games"`. The comparison is HTML:
`make viewer` serves the results tree (`platypus.viewer`), so it opens at
`http://localhost:<port>/<user>/sft_eval/<run>/compare_5games/index.html`.